# Medical Robotics Project Setup — Endoscapes

Notebook base in versione **Jupyter-friendly** per confrontare due pipeline:

1. **YOLO + Depth Anything V2**
2. **U-Net + Depth Anything V2**

Dataset usato: **Endoscapes-Seg50**.

Obiettivo:
- ricavare una **tool mask binaria** dal dataset Endoscapes;
- stimare una mappa di profondità con **Depth Anything V2**;
- generare una **contact mask** come output derivato;
- confrontare **YOLO** e **U-Net** sulla segmentazione dello strumento e sulla contact mask derivata.

## Struttura attesa del dataset

```text
Endoscapes/
├── train_seg/
│   ├── *.jpg
│   └── annotation_coco.json
├── val_seg/
│   ├── *.jpg
│   └── annotation_coco.json
├── test_seg/
│   ├── *.jpg
│   └── annotation_coco.json
├── semseg/
│   ├── *.png
│   └── ...
├── seg_label_map.txt
└── ...
```

Useremo i frame in `train_seg`, `val_seg`, `test_seg` e le maschere semantiche in `semseg/`.

## 1. Import e funzioni

In [ ]:
import json
import pandas as pd

from pathlib import Path

from transformers import pipeline
from PIL import Image
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def resolve_device(device=None):
    if device is not None:
        return device
    return 'cuda' if torch.cuda.is_available() else 'cpu'


def normalize_depth(depth):
    depth = depth.astype(np.float32)
    depth = depth - depth.min()
    if depth.max() > 0:
        depth = depth / depth.max()
    return depth


def overlay_mask(image_bgr, mask, color=(0, 255, 255), alpha=0.45):
    overlay = image_bgr.copy()
    overlay[mask > 0] = color
    return cv2.addWeighted(overlay, alpha, image_bgr, 1 - alpha, 0)


def save_image(path, image):
    ensure_dir(Path(path).parent)
    cv2.imwrite(str(path), image)


def to_tensor_image(image_bgr, image_size):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = cv2.resize(image_rgb, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    image_rgb = image_rgb.astype(np.float32) / 255.0
    image_rgb = np.transpose(image_rgb, (2, 0, 1))
    return torch.from_numpy(image_rgb).float()


def to_tensor_mask(mask, image_size):
    mask = cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.float32)
    mask = np.expand_dims(mask, axis=0)
    return torch.from_numpy(mask).float()

## config di base

In [ ]:
cfg = {
    'device': 'cuda',
    'image_size': 512,
    'batch_size': 2,
    'num_workers': 2,
    'output_dir': 'outputs',

    'data': {
        'root': 'endoscapes',
        'train_seg_dir': 'endoscapes/train_seg',
        'val_seg_dir': 'endoscapes/val_seg',
        'test_seg_dir': 'endoscapes/test_seg',
        'semseg_dir': 'endoscapes/semseg',
        'train_ann': 'endoscapes/train_seg/annotation_coco.json',
        'val_ann': 'endoscapes/val_seg/annotation_coco.json',
        'test_ann': 'endoscapes/test_seg/annotation_coco.json',
        'seg_label_map': 'endoscapes/seg_label_map.txt',
        'train_seg_vids': 'endoscapes/train_seg_vids.txt',
        'val_seg_vids': 'endoscapes/val_seg_vids.txt',
        'test_seg_vids': 'endoscapes/test_seg_vids.txt',
        'tool_class_name': 'Tool'
    },

    'yolo': {
        'weights': 'checkpoints/yolo11n-seg.pt',
        'conf': 0.25
    },

    'unet': {
        'checkpoint_path': 'checkpoints/unet_best.pth',
        'in_channels': 3,
        'out_channels': 1,
        'base_channels': 32,
        'threshold': 0.30
    },

    'depth': {
        'enabled': True,
        'encoder': 'vits',
        'checkpoint_path': 'checkpoints/depth_anything_v2_vits.pth'
    },

    'fusion': {
        'depth_threshold': 0.35,
        'min_area': 20
    }
}

## Dataset Endoscapes-Seg50

In [ ]:
def load_seg_label_map(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

    label_map = {}

    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            label = line.strip()

            if not label:
                continue

            if ':' in label:
                name, value = label.split(':', 1)
                label_map[name.strip()] = int(value.strip())
            else:
                label_map[label] = idx

    if len(label_map) == 0:
        raise ValueError(f"Nessuna label trovata in {path}")

    return label_map


def resolve_tool_class_id(label_map, tool_class_name):
    candidates = [
        tool_class_name,
        tool_class_name.lower(),
        tool_class_name.upper(),
        tool_class_name.capitalize()
    ]

    normalized = {k.lower(): v for k, v in label_map.items()}

    for cand in candidates:
        if cand in label_map:
            return cand, label_map[cand]
        if cand.lower() in normalized:
            real_key = [k for k in label_map.keys() if k.lower() == cand.lower()][0]
            return real_key, label_map[real_key]

    raise KeyError(
        f"Classe '{tool_class_name}' non trovata. Classi disponibili: {list(label_map.keys())}"
    )


label_map = load_seg_label_map(cfg['data']['seg_label_map'])
tool_class_name_resolved, tool_class_id = resolve_tool_class_id(
    label_map,
    cfg['data']['tool_class_name']
)

print("label_map:", label_map)
print("tool_class_name_resolved:", tool_class_name_resolved)
print("tool_class_id:", tool_class_id)


class EndoscapesSegDataset(Dataset):
    def __init__(self, split_dir, ann_path, semseg_dir, tool_class_id, image_size=512):
        self.split_dir = Path(split_dir)
        self.ann_path = Path(ann_path)
        self.semseg_dir = Path(semseg_dir)
        self.tool_class_id = tool_class_id
        self.image_size = image_size

        with open(self.ann_path, 'r') as f:
            self.coco = json.load(f)

        self.images = self.coco['images']
        self.file_names = sorted([img['file_name'] for img in self.images])

    def __len__(self):
        return len(self.file_names)

    def _load_tool_mask(self, stem):
        semseg_path = self.semseg_dir / f'{stem}.png'
        semseg = cv2.imread(str(semseg_path), cv2.IMREAD_UNCHANGED)

        if semseg is None:
            raise FileNotFoundError(f'Semantic mask not found: {semseg_path}')

        if semseg.ndim == 3:
            semseg = semseg[:, :, 0]

        return ((semseg == self.tool_class_id).astype(np.uint8) * 255)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        image_path = self.split_dir / file_name
        image_bgr = cv2.imread(str(image_path))

        if image_bgr is None:
            raise FileNotFoundError(f'Image not found: {image_path}')

        stem = Path(file_name).stem
        tool_mask = self._load_tool_mask(stem)

        return {
            'id': stem,
            'file_name': file_name,
            'image_bgr': image_bgr,
            'image': to_tensor_image(image_bgr, self.image_size),
            'tool_mask': to_tensor_mask(tool_mask, self.image_size),
            'tool_mask_fullres': tool_mask,
        }

## DataLoaders

In [ ]:
train_dataset = EndoscapesSegDataset(
    split_dir=cfg['data']['train_seg_dir'],
    ann_path=cfg['data']['train_ann'],
    semseg_dir=cfg['data']['semseg_dir'],
    tool_class_id=tool_class_id,
    image_size=cfg['image_size']
)

val_dataset = EndoscapesSegDataset(
    split_dir=cfg['data']['val_seg_dir'],
    ann_path=cfg['data']['val_ann'],
    semseg_dir=cfg['data']['semseg_dir'],
    tool_class_id=tool_class_id,
    image_size=cfg['image_size']
)

test_dataset = EndoscapesSegDataset(
    split_dir=cfg['data']['test_seg_dir'],
    ann_path=cfg['data']['test_ann'],
    semseg_dir=cfg['data']['semseg_dir'],
    tool_class_id=tool_class_id,
    image_size=cfg['image_size']
)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=cfg['num_workers'])
val_loader = DataLoader(val_dataset, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=cfg['num_workers'])

## U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetSmall(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.head(d1)

### U-Net inference stage

In [ ]:
class UNetStage:
    def __init__(self, checkpoint_path=None, in_channels=3, out_channels=1, base_channels=32, threshold=0.3, device=None):
        self.device = resolve_device(device)
        self.threshold = threshold

        self.model = UNetSmall(
            in_channels=in_channels,
            out_channels=out_channels,
            base_channels=base_channels
        ).to(self.device)

        if checkpoint_path and Path(checkpoint_path).exists():
            state = torch.load(checkpoint_path, map_location=self.device)
            if isinstance(state, dict) and 'model_state_dict' in state:
                state = state['model_state_dict']
            self.model.load_state_dict(state)
        else:
            print(f"[WARN] Checkpoint U-Net non trovato: {checkpoint_path}")

        self.model.eval()

    @torch.no_grad()
    def infer_logits(self, image_bgr, image_size=512):
        x = to_tensor_image(image_bgr, image_size).unsqueeze(0).to(self.device)
        return self.model(x)

    @torch.no_grad()
    def infer_mask(self, image_bgr, image_size=512, threshold=None, debug=False):
        if threshold is None:
            threshold = self.threshold

        h, w = image_bgr.shape[:2]
        logits = self.infer_logits(image_bgr, image_size=image_size)
        probs = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

        if debug:
            print("U-Net logits min/max/mean:", float(logits.min()), float(logits.max()), float(logits.mean()))
            print("U-Net probs min/max/mean:", float(probs.min()), float(probs.max()), float(probs.mean()))
            print("Pixels > thr:", float((probs > threshold).mean()))

        pred = (probs > threshold).astype(np.uint8) * 255
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
        return pred

## YOLO stage

In [ ]:
class YoloSegStage:
    def __init__(self, weights='checkpoints/yolo11n-seg.pt', conf=0.25, device=None):
        self.conf = conf
        self.device = resolve_device(device)
        self.model = YOLO(weights)

    def infer_mask(self, image_bgr, debug=False):
        results = self.model.predict(
            source=image_bgr,
            conf=self.conf,
            verbose=False,
            device=self.device,
        )
        result = results[0]
        h, w = image_bgr.shape[:2]

        if result.masks is None:
            if debug:
                print("[YOLO] no masks found")
            return np.zeros((h, w), dtype=np.uint8)

        masks = result.masks.data.detach().cpu().numpy()
        merged_mask = (np.any(masks > 0.5, axis=0)).astype(np.uint8) * 255
        merged_mask = cv2.resize(merged_mask, (w, h), interpolation=cv2.INTER_NEAREST)

        if debug:
            print("[YOLO] masks shape:", masks.shape, "coverage:", float((merged_mask > 0).mean()))

        return merged_mask

## DepthAnything v2

In [ ]:



class DepthAnythingV2Stage:
    def __init__(self, enabled=True, encoder='vits', checkpoint_path=None, device=None):
        self.enabled = enabled
        self.encoder = encoder
        self.checkpoint_path = checkpoint_path
        self.device = resolve_device(device)
        self.pipe = None

        if self.enabled:
            self._load_model()

    def _load_model(self):
        model_map = {
            'vits': 'depth-anything/Depth-Anything-V2-Small-hf',
            'vitb': 'depth-anything/Depth-Anything-V2-base-hf',
            'vitl': 'depth-anything/Depth-Anything-V2-Large-hf',
        }

        if self.encoder not in model_map:
            raise ValueError(f"Encoder non supportato: {self.encoder}. Usa vits, vitb o vitl.")

        self.pipe = pipeline(
            task="depth-estimation",
            model=model_map[self.encoder],
            device=0 if self.device == 'cuda' else -1
        )

    @torch.no_grad()
    def infer(self, image_bgr):
        h, w = image_bgr.shape[:2]

        if not self.enabled:
            return np.zeros((h, w), dtype=np.float32)

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image_pil = Image.fromarray(image_rgb)

        out = self.pipe(image_pil)
        depth = out["depth"]
        depth = np.asarray(depth, dtype=np.float32)

        if depth.shape != (h, w):
            depth = cv2.resize(depth, (w, h), interpolation=cv2.INTER_LINEAR)

        return depth


## Fusion stage

In [ ]:
class FusionStage:
    def __init__(self, depth_threshold=0.35, min_area=20):
        self.depth_threshold = depth_threshold
        self.min_area = min_area

    def fuse(self, tool_mask, depth_map, debug=False):
        depth_norm = normalize_depth(depth_map)
        contact_mask = ((tool_mask > 0) & (depth_norm < self.depth_threshold)).astype(np.uint8) * 255
        contact_mask = cv2.medianBlur(contact_mask, 5)

        if self.min_area > 0:
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
                (contact_mask > 0).astype(np.uint8), 8
            )
            filtered = np.zeros_like(contact_mask)
            for i in range(1, num_labels):
                if stats[i, cv2.CC_STAT_AREA] >= self.min_area:
                    filtered[labels == i] = 255
            contact_mask = filtered

        if debug:
            print("[Fusion] depth range:", float(depth_norm.min()), float(depth_norm.max()))
            print("[Fusion] contact coverage:", float((contact_mask > 0).mean()))

        return contact_mask

## Comparator

In [ ]:
class MedicalRoboticsComparator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = resolve_device(cfg.get('device', 'cuda'))
        self.output_dir = cfg.get('output_dir', 'outputs')
        ensure_dir(self.output_dir)

        self.image_size = cfg['image_size']
        self.yolo_stage = YoloSegStage(device=self.device, **cfg['yolo'])
        self.unet_stage = UNetStage(device=self.device, **cfg['unet'])
        self.depth_stage = DepthAnythingV2Stage(device=self.device, **cfg['depth'])
        self.fusion_stage = FusionStage(**cfg['fusion'])

    def infer_depth_once(self, image_bgr):
        return self.depth_stage.infer(image_bgr)

    def run_yolo_depth(self, image_bgr, depth_map=None, debug=False):
        if depth_map is None:
            depth_map = self.infer_depth_once(image_bgr)

        tool_mask = self.yolo_stage.infer_mask(image_bgr, debug=debug)
        contact_mask = self.fusion_stage.fuse(tool_mask, depth_map, debug=debug)
        return tool_mask, depth_map, contact_mask

    def run_unet_depth(self, image_bgr, depth_map=None, debug=False):
        if depth_map is None:
            depth_map = self.infer_depth_once(image_bgr)

        tool_mask = self.unet_stage.infer_mask(
            image_bgr,
            image_size=self.image_size,
            threshold=self.cfg['unet']['threshold'],
            debug=debug
        )
        contact_mask = self.fusion_stage.fuse(tool_mask, depth_map, debug=debug)
        return tool_mask, depth_map, contact_mask

    def run_both(self, image_bgr, debug=False):
        depth_map = self.infer_depth_once(image_bgr)

        yolo_tool = self.yolo_stage.infer_mask(image_bgr, debug=debug)
        unet_tool = self.unet_stage.infer_mask(
            image_bgr,
            image_size=self.image_size,
            threshold=self.cfg['unet']['threshold'],
            debug=debug
        )

        yolo_contact = self.fusion_stage.fuse(yolo_tool, depth_map, debug=debug)
        unet_contact = self.fusion_stage.fuse(unet_tool, depth_map, debug=debug)

        return {
            'depth_map': depth_map,
            'yolo_tool': yolo_tool,
            'yolo_contact': yolo_contact,
            'unet_tool': unet_tool,
            'unet_contact': unet_contact,
        }

## Metrics

In [ ]:
def binary_iou(pred_mask, gt_mask):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0
    return intersection / union


def binary_dice(pred_mask, gt_mask, eps=1e-7):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)

    intersection = np.logical_and(pred, gt).sum()
    return (2 * intersection + eps) / (pred.sum() + gt.sum() + eps)


def binary_precision(pred_mask, gt_mask, eps=1e-7):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)

    tp = np.logical_and(pred == 1, gt == 1).sum()
    fp = np.logical_and(pred == 1, gt == 0).sum()
    return (tp + eps) / (tp + fp + eps)


def binary_recall(pred_mask, gt_mask, eps=1e-7):
    pred = (pred_mask > 0).astype(np.uint8)
    gt = (gt_mask > 0).astype(np.uint8)

    tp = np.logical_and(pred == 1, gt == 1).sum()
    fn = np.logical_and(pred == 0, gt == 1).sum()
    return (tp + eps) / (tp + fn + eps)

## Evaluation

In [ ]:
def evaluate_comparison(dataset, comparator, max_samples=None):
    rows = []

    for idx in range(len(dataset)):
        if max_samples is not None and idx >= max_samples:
            break

        sample = dataset[idx]
        sample_id = sample['id']
        image_bgr = sample['image_bgr']
        gt_tool_mask = sample['tool_mask_fullres']

        out = comparator.run_both(image_bgr)

        yolo_tool = out['yolo_tool']
        yolo_contact = out['yolo_contact']
        unet_tool = out['unet_tool']
        unet_contact = out['unet_contact']

        rows.append({
            'id': sample_id,
            'gt_area': float((gt_tool_mask > 0).mean()),
            'yolo_area': float((yolo_tool > 0).mean()),
            'unet_area': float((unet_tool > 0).mean()),
            'yolo_contact_area': float((yolo_contact > 0).mean()),
            'unet_contact_area': float((unet_contact > 0).mean()),
            'yolo_tool_iou': binary_iou(yolo_tool, gt_tool_mask),
            'yolo_tool_dice': binary_dice(yolo_tool, gt_tool_mask),
            'yolo_tool_precision': binary_precision(yolo_tool, gt_tool_mask),
            'yolo_tool_recall': binary_recall(yolo_tool, gt_tool_mask),
            'unet_tool_iou': binary_iou(unet_tool, gt_tool_mask),
            'unet_tool_dice': binary_dice(unet_tool, gt_tool_mask),
            'unet_tool_precision': binary_precision(unet_tool, gt_tool_mask),
            'unet_tool_recall': binary_recall(unet_tool, gt_tool_mask),
        })

    return rows


def summarize_results(results):
    if len(results) == 0:
        return {}

    df = pd.DataFrame(results)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return df[numeric_cols].mean().to_dict()

## Run finale

In [ ]:
comparator = MedicalRoboticsComparator(cfg)

print("torch.cuda.is_available():", torch.cuda.is_available())
print("Comparator device:", comparator.device)
print("UNet device:", next(comparator.unet_stage.model.parameters()).device)
print("YOLO device request:", comparator.yolo_stage.device)

results = evaluate_comparison(test_dataset, comparator, max_samples=10)
summary = summarize_results(results)

print('num samples:', len(results))
print(summary)

sample_debug = test_dataset[0]
print("\n=== Debug sample ===")
_ = comparator.yolo_stage.infer_mask(sample_debug['image_bgr'], debug=True)
_ = comparator.unet_stage.infer_mask(sample_debug['image_bgr'], image_size=cfg['image_size'], debug=True)

## Tabelle di confronto


In [ ]:
results_df = pd.DataFrame(results)

comparison_df = pd.DataFrame([
    {
        'Model': 'YOLO + Depth Anything V2',
        'Mean IoU': results_df['yolo_tool_iou'].mean(),
        'Std IoU': results_df['yolo_tool_iou'].std(),
        'Mean Dice': results_df['yolo_tool_dice'].mean(),
        'Std Dice': results_df['yolo_tool_dice'].std(),
        'Mean Precision': results_df['yolo_tool_precision'].mean(),
        'Mean Recall': results_df['yolo_tool_recall'].mean(),
        'Mean Tool Area': results_df['yolo_area'].mean(),
        'Mean Contact Area': results_df['yolo_contact_area'].mean(),
        'Best IoU': results_df['yolo_tool_iou'].max(),
        'Worst IoU': results_df['yolo_tool_iou'].min(),
    },
    {
        'Model': 'U-Net + Depth Anything V2',
        'Mean IoU': results_df['unet_tool_iou'].mean(),
        'Std IoU': results_df['unet_tool_iou'].std(),
        'Mean Dice': results_df['unet_tool_dice'].mean(),
        'Std Dice': results_df['unet_tool_dice'].std(),
        'Mean Precision': results_df['unet_tool_precision'].mean(),
        'Mean Recall': results_df['unet_tool_recall'].mean(),
        'Mean Tool Area': results_df['unet_area'].mean(),
        'Mean Contact Area': results_df['unet_contact_area'].mean(),
        'Best IoU': results_df['unet_tool_iou'].max(),
        'Worst IoU': results_df['unet_tool_iou'].min(),
    }
])

comparison_df['Overall Score'] = (comparison_df['Mean IoU'] + comparison_df['Mean Dice']) / 2
comparison_df['Rank'] = comparison_df['Overall Score'].rank(ascending=False, method='min').astype(int)
comparison_df = comparison_df.sort_values('Rank').reset_index(drop=True)

results_df['iou_winner'] = np.where(
    results_df['yolo_tool_iou'] > results_df['unet_tool_iou'],
    'YOLO',
    np.where(
        results_df['yolo_tool_iou'] < results_df['unet_tool_iou'],
        'U-Net',
        'Tie'
    )
)

results_df['dice_winner'] = np.where(
    results_df['yolo_tool_dice'] > results_df['unet_tool_dice'],
    'YOLO',
    np.where(
        results_df['yolo_tool_dice'] < results_df['unet_tool_dice'],
        'U-Net',
        'Tie'
    )
)

results_df['iou_gap'] = results_df['yolo_tool_iou'] - results_df['unet_tool_iou']
results_df['dice_gap'] = results_df['yolo_tool_dice'] - results_df['unet_tool_dice']

print("\n=== Comparison Table ===")
display(comparison_df.round(4))

print("\n=== Per-sample Table ===")
display(results_df.round(4))

print("\n=== Best / Worst cases by IoU gap ===")
display(
    results_df.sort_values('iou_gap', ascending=False)[
        ['id', 'yolo_tool_iou', 'unet_tool_iou', 'iou_gap', 'iou_winner']
    ].head(5).round(4)
)

display(
    results_df.sort_values('iou_gap', ascending=True)[
        ['id', 'yolo_tool_iou', 'unet_tool_iou', 'iou_gap', 'iou_winner']
    ].head(5).round(4)
)

## Grafico confronto metriche

In [ ]:

plot_df = comparison_df[['Model', 'Mean IoU', 'Mean Dice']].copy()
plot_df = plot_df.set_index('Model')

ax = plot_df.plot(kind='bar', figsize=(10, 5), rot=0)
ax.set_title('Model comparison on tool segmentation')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
sample = test_dataset[0]
image_bgr = sample['image_bgr']
gt_tool_mask = sample['tool_mask_fullres']

out = comparator.run_both(image_bgr)

yolo_tool = out['yolo_tool']
yolo_depth = out['depth_map']
yolo_contact = out['yolo_contact']
unet_tool = out['unet_tool']
unet_depth = out['depth_map']
unet_contact = out['unet_contact']

fig, axes = plt.subplots(2, 5, figsize=(22, 10))

axes[0, 0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Input')
axes[0, 1].imshow(gt_tool_mask, cmap='gray')
axes[0, 1].set_title('GT tool mask')
axes[0, 2].imshow(yolo_tool, cmap='gray')
axes[0, 2].set_title('YOLO tool mask')
axes[0, 3].imshow(normalize_depth(yolo_depth), cmap='plasma')
axes[0, 3].set_title('YOLO depth')
axes[0, 4].imshow(yolo_contact, cmap='gray')
axes[0, 4].set_title('YOLO+Depth contact')

axes[1, 0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Input')
axes[1, 1].imshow(gt_tool_mask, cmap='gray')
axes[1, 1].set_title('GT tool mask')
axes[1, 2].imshow(unet_tool, cmap='gray')
axes[1, 2].set_title('U-Net tool mask')
axes[1, 3].imshow(normalize_depth(unet_depth), cmap='plasma')
axes[1, 3].set_title('U-Net depth')
axes[1, 4].imshow(unet_contact, cmap='gray')
axes[1, 4].set_title('U-Net+Depth contact')

for ax in axes.ravel():
    ax.axis('off')

plt.tight_layout()
plt.show()